# Lab 05 · Aprendizaje no supervisado

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 5*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Qué encuentra un algoritmo cuando nadie le dice qué buscar</li>
<li>Cómo se decide en cuántos grupos partir algo, si no hay respuesta correcta</li>
<li>Cómo se mira una tabla de veinticuatro columnas en una hoja de papel</li>
<li>Qué hacer con lo que no cae en ningún grupo</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Preparar una tabla para agrupar, con la normalización que corresponda</li>
<li>Agrupar con K-Means y justificar el número de grupos con el codo y la silueta</li>
<li>Reducir veinticuatro dimensiones a dos con PCA sin perder lo que importa</li>
<li>Separar grupos densos de ruido con DBSCAN, y saber qué significa ese ruido</li>
<li>Encontrar temas en texto libre sin haberlo etiquetado antes</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por región, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaría una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    with open("precios_nudo.json","w") as f:
        json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                               "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
                   "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
                  f)

    # ------------------------------------------------ bitacora de mantenimiento
    # Seiscientos eventos del año, con dos patrones plantados a propósito que el
    # lab va a tener que encontrar y medir.
    rngm = np.random.default_rng(404)
    TIPOS = ["preventivo","correctivo","falla_electrica","falla_mecanica",
             "evento_climatico","inspeccion"]
    PESOS = {"hidro":[.34,.14,.12,.16,.02,.22], "solar":[.38,.12,.16,.08,.02,.24],
             "eolica":[.26,.12,.12,.20,.08,.22], "gas":[.30,.18,.14,.18,.01,.19],
             "carbon":[.28,.18,.14,.20,.01,.19], "diesel":[.14,.52,.10,.12,.01,.11]}
    DURA = {"preventivo":(4,24), "correctivo":(6,72), "falla_electrica":(2,48),
            "falla_mecanica":(8,96), "evento_climatico":(3,36), "inspeccion":(1,8)}
    n_dias = len(fechas)
    invierno = np.isin(fechas.month.to_numpy(), [5,6,7,8])
    peso_clima = np.where(invierno, 4.0, 1.0); peso_clima /= peso_clima.sum()

    ev = []
    for _,c in centrales.iterrows():
        p = np.array(PESOS[c.tecnologia]); p = p/p.sum()
        n = 30 if c.tecnologia=="diesel" else 22
        for tipo, d in zip(rngm.choice(TIPOS, size=n, p=p), rngm.integers(0, n_dias, size=n)):
            ev.append([c.central, int(d), str(tipo)])
        n_cl = 12 if c.tecnologia=="eolica" else 1
        for d in rngm.choice(n_dias, size=n_cl, replace=False, p=peso_clima):
            ev.append([c.central, int(d), "evento_climatico"])

    # Patron 1, el 70 por ciento de las fallas electricas arrastra un correctivo
    # en la misma central dentro de los tres dias siguientes.
    elec = [i for i,e in enumerate(ev) if e[2]=="falla_electrica"]
    for i in sorted(rngm.choice(elec, size=int(round(len(elec)*0.70)), replace=False)):
        ev.append([ev[i][0], min(ev[i][1] + int(rngm.integers(0,4)), n_dias-1), "correctivo"])
    # Patron 2, la mitad de los eventos climaticos trae una falla mecanica el mismo dia.
    clim = [i for i,e in enumerate(ev) if e[2]=="evento_climatico"]
    for i in sorted(rngm.choice(clim, size=int(round(len(clim)*0.50)), replace=False)):
        ev.append([ev[i][0], ev[i][1], "falla_mecanica"])

    nombres = centrales["central"].to_numpy()
    while len(ev) < 600:
        ev.append([str(nombres[int(rngm.integers(0,len(nombres)))]),
                   int(rngm.integers(0,n_dias)),
                   "preventivo" if rngm.random()<0.6 else "inspeccion"])
    ev = ev[:600]

    mant = pd.DataFrame([{"central":c, "fecha":fechas[d].strftime("%Y-%m-%d"),
                          "tipo_evento":t,
                          "duracion_horas":int(rngm.integers(DURA[t][0], DURA[t][1]+1))}
                         for c,d,t in ev])

    # ------------------------------------------- texto libre de cada evento
    # Una observación escrita como la escribiría el turno, con vocabulario
    # propio de cada tipo. El Módulo 5 busca temas ahí adentro.
    VOCAB = {
     "falla_electrica": (["el transformador de poder","el interruptor principal",
        "la barra de media tensión","el relé de protección","el aislador de línea"],
        ["sobretensión sostenida","un cortocircuito monofásico","corriente de fuga elevada",
         "el disparo de la protección diferencial"],
        ["se aísla el circuito y se normaliza la tensión","se reemplaza el relé y se recalibra",
         "se reconecta el interruptor tras verificar la aislación"]),
     "falla_mecanica": (["el rodamiento del eje","la caja multiplicadora","el acoplamiento",
        "el sello del descanso","la bomba de lubricación"],
        ["vibración fuera de norma","temperatura elevada en el descanso","ruido anormal",
         "pérdida de aceite"],
        ["se reemplaza el rodamiento y se alinea el eje","se rellena y se purga el circuito de aceite",
         "se ajusta el acoplamiento y se mide la vibración"]),
     "evento_climatico": (["la línea de evacuación","el patio de alta tensión",
        "el camino de acceso","la estructura de la torre","el pararrayos del patio"],
        ["viento sobre lo previsto","una descarga atmosférica cercana","acumulación de nieve",
         "lluvia intensa con anegamiento"],
        ["se inspecciona la estructura y se despeja la faja","se repone el servicio al amainar",
         "se drena el sector y se revisa la puesta a tierra"]),
     "preventivo": (["el sistema de refrigeración","los filtros de aire","el tablero de control",
        "las conexiones de fuerza","el grupo hidráulico"],
        ["la mantención programada","el cambio de filtros","el ajuste de rutina",
         "la lubricación periódica"],
        ["se cambian filtros y se registra la lectura","se reaprietan las conexiones y se sella",
         "se completa la pauta sin observaciones"]),
     "correctivo": (["el equipo afectado","la unidad detenida","el componente dañado",
        "la sección fuera de servicio","el módulo de potencia"],
        ["la reparación de la falla del turno anterior","el levantamiento de la indisponibilidad",
         "la orden de trabajo pendiente","la intervención de emergencia"],
        ["se repara y se devuelve a servicio","se reemplaza la pieza y se prueba en vacío",
         "se normaliza y se informa al despacho"]),
     "inspeccion": (["el conjunto de medida","la señalética del área","los niveles de aceite",
        "el estado de los accesos","el registro de alarmas"],
        ["la ronda de rutina","la verificación visual","la lectura de instrumentos",
         "el chequeo de seguridad"],
        ["se deja constancia sin hallazgos","se anota una observación menor",
         "se programa revisión de detalle"]),
    }
    PLANT = ["Se registra {s} sobre {c}.",
             "Se detecta {s} en {c}, {a}.",
             "El operador reporta {s}, se revisa {c} y {a}.",
             "Evento por {s} en {c}, {a}."]
    def _obs(t):
        c, s, a = (VOCAB[t][k][int(rngm.integers(0, len(VOCAB[t][k])))] for k in (0, 1, 2))
        return PLANT[int(rngm.integers(0, len(PLANT)))].format(s=s, c=c, a=a)
    mant["observacion"] = [_obs(t) for t in mant["tipo_evento"]]
    mant.sort_values(["fecha","central"], kind="stable").reset_index(drop=True) \
        .to_csv("mantenimiento.csv", index=False)
print("Datos listos, incluida la bitácora con sus observaciones")


## 1. Preparar los vectores

In [ ]:
import pandas as pd

# Cada central como un vector de 24 números, su día promedio. Igual que en el Lab 04.
gen = pd.read_csv("generacion.csv", parse_dates=["fecha"])
centrales = pd.read_csv("centrales.csv")
perfil = gen.pivot_table(index="central", columns="hora", values="mwh", aggfunc="mean")

print(perfil.shape)
print(perfil.iloc[:3, 10:15].round(1))

In [ ]:
# El problema de siempre. La Costa Brava genera 300 MWh a las 12 y la Vega Azul 50.
print(perfil.max(axis=1).sort_values(ascending=False).round(1).to_string())

In [ ]:
# Primera normalización, dividir por la potencia instalada. Ahora cada número
# es un factor de planta horario, entre cero y uno, y todas son comparables.
potencia = centrales.set_index("central")["potencia_mw"]
factor = perfil.div(potencia, axis=0)

print("antes, rango de la tabla ", round(perfil.values.min(), 1), "a", round(perfil.values.max(), 1))
print("después, rango de la tabla", round(factor.values.min(), 3), "a", round(factor.values.max(), 3))

In [ ]:
# Así se ve una central después de normalizar. Es su forma, no su tamaño.
print(factor.loc["Parque Solar Altiplano"].round(2).to_string())

In [ ]:
from sklearn.preprocessing import StandardScaler

# Segunda normalización, la de la biblioteca. Deja cada columna con promedio
# cero y desviación uno, que es lo que esperan casi todos estos algoritmos.
escalador = StandardScaler()
X = escalador.fit_transform(factor)

print(X.shape)
print("promedio de X", round(float(X.mean()), 6), " desviación", round(float(X.std()), 3))

In [ ]:
#@title Dos normalizaciones distintas, y las dos hacen falta { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Dos normalizaciones distintas, y las dos hacen falta</div><p>Dividir por la potencia instalada es una decisión del negocio. Saca del medio el tamaño de la central, que para esta pregunta no interesa, y deja el factor de planta, que es comparable entre una central de 480 MW y una de 45.</p>
<p>El <code>StandardScaler</code> es una decisión técnica. Deja cada hora con promedio cero y desviación uno, para que ninguna hora pese más que otra solo porque sus números son más grandes. Sin eso, las horas del mediodía dominarían la distancia.</p>
<p>Son cosas distintas y las dos se hacen. En el bloque 2 vamos a ver qué pasa si uno se salta la primera.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Mira una central antes y después</strong>
<p>Compara el perfil de <code>Carboelectrica Costa Brava</code> con el de <code>Central Vega Azul</code>, primero en la tabla <code>perfil</code> y después en la tabla <code>factor</code>. Son la central más grande y una de las más chicas. Fíjate en qué cambia y qué no.</p></div>"""))

In [ ]:
# Tu turno
# Cambia perfil por factor y vuelve a mirar las dos filas.
dos = ["Carboelectrica Costa Brava", "Central Vega Azul"]

print(perfil.loc[dos, 10:14].round(2).to_string())

In [ ]:
#@title Hasta acá llega la primera clase { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Hasta acá llega la primera clase</div><p>Los datos quedaron listos. Veinte centrales, veinticuatro columnas, normalizadas dos veces, una por criterio del negocio y otra por requisito del algoritmo. Ese trabajo es el que decide si lo que viene funciona.</p>
<p>La próxima clase partimos pidiéndole al algoritmo que agrupe estas veinte centrales sin decirle cuántos grupos hay ni cómo se llaman.</p>
<p><strong>Cuando vuelvas, ejecuta el notebook desde arriba.</strong> Entorno de ejecución, Ejecutar todo. Colab no guarda el estado entre sesiones.</p></div>"""))

## 2. K-Means

In [ ]:
from sklearn.cluster import KMeans

# K-Means pide el número de grupos de entrada. Ese es su límite y su decisión.
km = KMeans(n_clusters=4, random_state=2026, n_init=10).fit(X)

print(pd.Series(km.labels_, index=factor.index).sort_values().to_string())

In [ ]:
# La tabla que importa. Los grupos que encontró contra la tecnología real,
# que el algoritmo nunca vio.
tecnologia = centrales.set_index("central")["tecnologia"]

print(pd.crosstab(tecnologia[factor.index], km.labels_).to_string())

In [ ]:
# Cómo se elige el número de grupos. La inercia es lo que queda sin explicar,
# y siempre baja al agregar grupos. Se busca el codo, donde deja de bajar fuerte.
inercias = []
for k in range(1, 9):
    inercias.append(KMeans(n_clusters=k, random_state=2026, n_init=10).fit(X).inertia_)

for k, v in zip(range(1, 9), inercias):
    print(k, round(v, 1))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, 9), inercias, marker="o")
ax.set_title("El codo está donde la curva deja de caer")
ax.set_xlabel("número de grupos")
ax.set_ylabel("inercia")
plt.show()

In [ ]:
# Con seis grupos pasa algo que vale la pena mirar.
km6 = KMeans(n_clusters=6, random_state=2026, n_init=10).fit(X)

print(pd.crosstab(tecnologia[factor.index], km6.labels_).to_string())

In [ ]:
# Y ahora la prueba que justifica todo el bloque 1. Lo mismo, pero sin haber
# dividido por la potencia instalada.
X_crudo = StandardScaler().fit_transform(perfil)
km_crudo = KMeans(n_clusters=6, random_state=2026, n_init=10).fit(X_crudo)

print(pd.crosstab(tecnologia[perfil.index], km_crudo.labels_).to_string())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Los centros de cada grupo</strong>
<p>Los centros están en <code>km6.cluster_centers_</code>, pero en el espacio escalado. Devuélvelos al original con <code>escalador.inverse_transform</code> y mira el perfil promedio de cada grupo. Pista, quedan como una matriz de 6 por 24.</p></div>"""))

In [ ]:
# Tu turno
# Devuelve los centros al espacio original y mira las horas 10 a 14.
centros = km6.cluster_centers_

print(pd.DataFrame(centros).iloc[:, 10:15].round(2).to_string())

## 3. PCA

In [ ]:
from sklearn.decomposition import PCA

# PCA busca las direcciones donde los datos más se estiran, y las ordena.
pca = PCA().fit(X)

print("cuánto explica cada componente")
print(pd.Series(pca.explained_variance_ratio_[:6]).round(3).to_string())

In [ ]:
# Con dos componentes ya está casi todo. Eso es lo que permite dibujarlo.
acumulada = pca.explained_variance_ratio_.cumsum()

print("con 1 componente ", round(acumulada[0], 3))
print("con 2 componentes", round(acumulada[1], 3))
print("con 3 componentes", round(acumulada[2], 3))

In [ ]:
# Las veinte centrales en dos dimensiones, coloreadas por tecnología.
Z = PCA(n_components=2).fit_transform(X)
mapa = pd.DataFrame(Z, columns=["pc1", "pc2"], index=factor.index)
mapa["tecnologia"] = tecnologia[mapa.index].values

print(mapa.groupby("tecnologia")[["pc1", "pc2"]].mean().round(2).to_string())

In [ ]:
COLORES = {"hidro": "tab:blue", "solar": "tab:orange", "eolica": "tab:green",
           "gas": "tab:red", "carbon": "tab:brown", "diesel": "tab:gray"}

fig, ax = plt.subplots(figsize=(7, 5))
for t, g in mapa.groupby("tecnologia"):
    ax.scatter(g["pc1"], g["pc2"], label=t, s=45, alpha=0.75, color=COLORES[t])
ax.set_title("Las veinte centrales en dos dimensiones\n(las de una misma tecnología quedan casi encima)")
ax.set_xlabel("componente 1")
ax.set_ylabel("componente 2")
ax.legend()
plt.show()

In [ ]:
# Qué es la componente 1. Se mira su peso en cada una de las 24 horas.
pesos = pd.Series(PCA(n_components=2).fit(X).components_[0], index=factor.columns)

print(pesos.round(2).to_string())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>La tercera componente</strong>
<p>Pide tres componentes en vez de dos y mira cuánto agrega la tercera. Con ese número, decide si vale la pena dibujarla. Pista, <code>explained_variance_ratio_</code> ya te lo dice.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 2 por un 3 y mira la varianza explicada de cada componente.
p2 = PCA(n_components=2).fit(X)

print(p2.explained_variance_ratio_.round(4))

## 4. DBSCAN

In [ ]:
# Cambiamos de unidad. Ya no una fila por central, sino una por central y día.
perfil_dia = gen.pivot_table(index=["central", "fecha"], columns="hora",
                             values="mwh", aggfunc="mean")
potencia_dia = perfil_dia.index.get_level_values("central").map(potencia)
factor_dia = perfil_dia.div(potencia_dia, axis=0)
X_dia = StandardScaler().fit_transform(factor_dia)

print(X_dia.shape, "veinte centrales por 366 días")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# DBSCAN no pide número de grupos, pide un radio. Para elegirlo se mira a qué
# distancia está el quinto vecino de cada punto, ordenado de menor a mayor.
distancias, _ = NearestNeighbors(n_neighbors=5).fit(X_dia).kneighbors(X_dia)
quinto = np.sort(distancias[:, -1])

for p in (50, 75, 90, 95, 99):
    print(f"percentil {p}: {np.percentile(quinto, p):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(quinto)
ax.set_title("Distancia al quinto vecino, ordenada")
ax.set_xlabel("puntos")
ax.set_ylabel("distancia")
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

# El salto de la curva está entre 0,7 y 2,7. Probamos a los dos lados.
for eps in (1.5, 2.0, 2.5, 3.0):
    db = DBSCAN(eps=eps, min_samples=5).fit(X_dia)
    grupos = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    ruido = int((db.labels_ == -1).sum())
    print(f"eps {eps}: {grupos} grupos, {ruido} puntos de ruido"
          f" ({100 * ruido / len(X_dia):.1f} por ciento)")

In [ ]:
# Con radio 1,5 quedan 1.464 puntos fuera de todo grupo. Quiénes son.
db = DBSCAN(eps=1.5, min_samples=5).fit(X_dia)
tecnologia_dia = perfil_dia.index.get_level_values("central").map(tecnologia)

print(pd.crosstab(tecnologia_dia, db.labels_).to_string())

In [ ]:
# Mil cuatrocientos sesenta y cuatro son cuatro centrales por 366 días.
print("días eólicos en la tabla", int((tecnologia_dia == "eolica").sum()))
print("puntos marcados como ruido", int((db.labels_ == -1).sum()))

In [ ]:
#@title El ruido no siempre es un error { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">El ruido no siempre es un error</div><p>DBSCAN dejó fuera de todo grupo a los 1.464 días eólicos, que son las cuatro eólicas por los 366 días del año. No es una falla del algoritmo.</p>
<p>Un punto queda en un grupo si tiene al menos cinco vecinos dentro del radio. El día de un parque eólico casi nunca se parece a otro día, porque el viento no repite forma. Los días de una solar o de una térmica sí se parecen entre ellos, y por eso forman grupos densos.</p>
<p>O sea que el ruido de DBSCAN acá es un hallazgo, no un descarte. Dice cuáles son las centrales cuya producción no tiene un día tipo.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Sube el radio hasta que el ruido desaparezca</strong>
<p>Prueba <code>eps</code> en 3,5 y mira qué pasa con el ruido y con el número de grupos. Después decide cuál de los dos radios usarías y por qué.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 1.5 por 3.5 y compara el ruido y los grupos.
prueba = DBSCAN(eps=1.5, min_samples=5).fit(X_dia)

print("grupos", len(set(prueba.labels_)) - (1 if -1 in prueba.labels_ else 0),
      " ruido", int((prueba.labels_ == -1).sum()))

## 5. Temas en el texto libre

In [ ]:
# La bitácora del Módulo 4, ahora con la columna que no habíamos usado.
mant = pd.read_csv("mantenimiento.csv", parse_dates=["fecha"])

muestra = mant[["tipo_evento", "observacion"]].head(3).copy()
muestra["observacion"] = muestra["observacion"].str.slice(0, 58) + "..."

print(muestra.to_string(index=False))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Las palabras que aparecen en todas partes no distinguen nada, se sacan.
VACIAS = ["el", "la", "los", "las", "un", "una", "de", "del", "en", "y", "se",
          "que", "por", "con", "sobre", "tras", "su", "sus", "al", "lo", "es",
          "durante", "ronda", "turno", "registra", "detecta", "reporta",
          "evento", "operador", "revisa"]
vectorizador = CountVectorizer(stop_words=VACIAS, max_df=0.9, min_df=3)
matriz = vectorizador.fit_transform(mant["observacion"])

print(matriz.shape, "seiscientos textos por", len(vectorizador.get_feature_names_out()), "palabras")

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA busca temas. No sabe que existen seis tipos de evento, le pedimos seis temas.
lda = LatentDirichletAllocation(n_components=6, random_state=2026).fit(matriz)
palabras = vectorizador.get_feature_names_out()

for i, tema in enumerate(lda.components_):
    top = [palabras[j] for j in tema.argsort()[-7:][::-1]]
    print(f"tema {i}:", ", ".join(top))

In [ ]:
# Y ahora la prueba. A cada texto se le asigna su tema dominante, y se cruza
# contra el tipo de evento, que LDA nunca vio.
dominante = lda.transform(matriz).argmax(axis=1)
cruce = pd.crosstab(mant["tipo_evento"], dominante)

print(cruce.to_string())

In [ ]:
# Cuánto acertó, si uno le pide a cada tema que represente un tipo.
print("textos en el tema mayoritario de su tipo", int(cruce.max(axis=1).sum()), "de", len(mant))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Pide tres temas en vez de seis</strong>
<p>Vuelve a ajustar el LDA con <code>n_components=3</code> y mira qué tipos de evento se juntan. Los que caen en el mismo tema son los que comparten vocabulario.</p></div>"""))

In [ ]:
# Tu turno
# Cambia el 6 por un 3 y vuelve a cruzar contra tipo_evento.
lda_prueba = LatentDirichletAllocation(n_components=6, random_state=2026).fit(matriz)

print(pd.crosstab(mant["tipo_evento"],
                  lda_prueba.transform(matriz).argmax(axis=1)).to_string())

## 6. Evaluar sin respuesta correcta

In [ ]:
from sklearn.metrics import silhouette_score

# La silueta mide, para cada punto, qué tan cerca está de su grupo comparado
# con el grupo vecino. Va de menos uno a uno, y no necesita la respuesta.
for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, random_state=2026, n_init=10).fit_predict(X)
    print(f"k={k}  silueta {silhouette_score(X, etiquetas):.3f}")

In [ ]:
# La silueta y el codo no siempre apuntan al mismo número. Acá hay que mirarlos
# juntos, con la inercia al lado.
resumen = pd.DataFrame({"inercia": [round(v, 1) for v in inercias[1:7]],
                        "silueta": [round(silhouette_score(
                            X, KMeans(n_clusters=k, random_state=2026,
                                      n_init=10).fit_predict(X)), 3)
                                    for k in range(2, 8)]},
                       index=range(2, 8))

print(resumen.to_string())

In [ ]:
# Y la prueba que en el trabajo real no existe, comparar contra la tecnología.
km5 = KMeans(n_clusters=5, random_state=2026, n_init=10).fit(X)

print(pd.crosstab(tecnologia[factor.index], km5.labels_).to_string())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>La silueta sobre otro agrupamiento</strong>
<p>Calcula la silueta de la solución de DBSCAN con <code>eps=3.0</code>, sacando antes los puntos de ruido. Compárala con la de K-Means. Pista, la silueta no acepta la etiqueta menos uno.</p></div>"""))

In [ ]:
# Tu turno
# Saca el ruido antes de calcular la silueta de DBSCAN.
db3 = DBSCAN(eps=3.0, min_samples=5).fit(X_dia)
sin_ruido = db3.labels_ != -1

print("puntos que quedan", int(sin_ruido.sum()), "de", len(X_dia))

## 7. Contra la verdad

In [ ]:
#@title Lo que este bloque no se puede hacer en la vida real { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Lo que este bloque no se puede hacer en la vida real</div><p>Igual que en los dos módulos anteriores, acá tenemos la respuesta y en el trabajo no la vas a tener. Sabemos la tecnología de cada central, y ninguno de los algoritmos la vio.</p>
<p>Eso permite hacer la pregunta que de verdad importa. Lo que el algoritmo encontró solo, ¿se parece a algo que ya sabíamos?</p></div>"""))

In [ ]:
# Los tres agrupamientos, contra la tecnología real.
from sklearn.metrics import adjusted_rand_score

verdad = tecnologia[factor.index].values
for nombre, etiquetas in [("K-Means k=4", km.labels_),
                          ("K-Means k=6", km6.labels_),
                          ("K-Means sin normalizar", km_crudo.labels_)]:
    print(f"{nombre:26s} índice de Rand ajustado {adjusted_rand_score(verdad, etiquetas):.3f}")

In [ ]:
#@title Un uno exacto acá, y una sospecha en el trabajo { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Un uno exacto acá, y una sospecha en el trabajo</div><p>El 1,000 de la segunda línea sale porque los datos son sintéticos y la tecnología determina por completo el perfil de cada central. Acá es lo esperable.</p>
<p>Con datos reales un uno sería motivo de sospecha, no de celebración. Lo primero que hay que revisar es si hubo una fuga de información, es decir si alguna variable de entrada traía la respuesta adentro.</p></div>"""))

In [ ]:
# Uno quiere decir que el agrupamiento reproduce exactamente la tecnología,
# y cero que no tiene nada que ver con ella.
print("con seis grupos y normalizando, el algoritmo recuperó las seis tecnologías")
print(pd.crosstab(tecnologia[factor.index], km6.labels_).to_string())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Y el LDA</strong>
<p>Calcula el índice de Rand ajustado entre el tema dominante del LDA y la columna <code>tipo_evento</code>. Es la misma pregunta sobre el texto, y el número va a ser bastante más bajo. Piensa por qué.</p></div>"""))

In [ ]:
# Tu turno
# Usa adjusted_rand_score entre tipo_evento y el tema dominante.
print("tipos distintos", mant["tipo_evento"].nunique(), " temas", len(set(dominante)))

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li>Antes de agrupar hay dos normalizaciones, una del negocio y otra del algoritmo, y las dos hacen falta</li>
<li>K-Means pide el número de grupos, y ese número se justifica con el codo y con la silueta, nunca solo con uno</li>
<li>PCA no agrupa, reduce, y acá dos componentes explican el 99,5 por ciento de lo que hay</li>
<li>DBSCAN no pide grupos, pide un radio, y lo que deja como ruido puede ser el hallazgo</li>
<li>LDA encuentra temas en texto libre sin que nadie lo haya etiquetado, y se equivoca donde el vocabulario se comparte</li>
<li>Sin respuesta correcta, la evaluación es una medida interna más el juicio de alguien que conoce el dominio</li>
</ul>
<p>En el Lab 06 vamos a contar todo esto con gráficos, que es lo que queda cuando el análisis ya está hecho.</p></div>"""))